In [0]:
# set whether normal process or being run by Elsevier supporters
permissions = 'fulldata' # default or fulldata
# set the project name
project_name = 'proj_050_teamscience'  # This is the folder name that you work in, e.g. 'test_project_1'

In [0]:
#Alick: I'll switch to the latest snapshot (v008) - wording there is confusing! 
#%run /Snippets/header_latest

In [0]:
%run /Snippets/header_008

## Python Libraries and Functions

In [0]:
# libraries 
from pyspark.sql.functions import col, count, explode, array, mean, stddev, filter, percentile_approx, arrays_zip, size, array_distinct, sum, reverse, collect_list, array_sort, posexplode, max, arrays_overlap, lit, when, cast
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import Window
from pyspark.ml.feature import StringIndexer

# make a DF to store everything
model_output_df = pd.DataFrame(columns = ["model", "coef", "se", "t", "p", "r2"])
descriptives_output_df = pd.DataFrame(columns = ["model", "subjareas", "mean", "sd", "sample", "median", "iqr_25", "iqr_75"])

# model summary print out 
def modelsummary(model, hyp_model, df):
    import numpy as np
    print ("Note: the last rows are the information for Intercept")
    print ("##","-------------------------------------------------")
    print ("##","  Estimate   |   Std.Error | t Values  |  P-value")
    coef = np.append(list(model.coefficients),model.intercept)
    Summary=model.summary

    for i in range(len(Summary.pValues)):
        print ("##",'{:10.6f}'.format(coef[i]),\
        '{:10.6f}'.format(Summary.coefficientStandardErrors[i]),\
        '{:8.3f}'.format(Summary.tValues[i]),\
        '{:10.6f}'.format(Summary.pValues[i]))
        # store the data 
        row = pd.DataFrame({"model": hyp_model, "coef": coef[i], "se": Summary.coefficientStandardErrors[i], 
               "t": Summary.tValues[i], "p": Summary.pValues[i], "r2": Summary.r2}, index = [0])
        df = pd.concat([df, row], ignore_index = True, axis = 0)

    print ("##",'---')
    print ("##","Mean squared error: % .6f" \
           % Summary.meanSquaredError, ", RMSE: % .6f" \
           % Summary.rootMeanSquaredError )
    print ("##","Multiple R-squared: %f" % Summary.r2, ", \
            Total iterations: %i"% Summary.totalIterations)
    return(df)

## Research Question 1a

### Descriptive Statistics

We will first calculate the mean, median, and interquartile range of team size by discipline. We will not explicitly compare discipline intercepts in a post hoc format with any form of test. Instead we will use visualization techniques to display the distribution and their interquartile range for readers to examine (i.e., a forest plot).

In [0]:
# load the ANI dataframe which contains author information 
df_ani = spark.read.format("parquet").load(basePath+tablename_ani)
df_ani.createOrReplaceTempView("df_ani")
df_ani.printSchema()
df_ani_smaller = df_ani.filter(df_ani.Year > 1969)\
  .filter(df_ani.publication_type == "ar")\
  .sample(False, .25, 8675309) 

df_ani_smaller.count()

# display(df_ani)

root
 |-- Eid: long (nullable = true)
 |-- Year: integer (nullable = true)
 |-- title: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- title: string (nullable = true)
 |    |    |-- lang: string (nullable = true)
 |    |    |-- original: string (nullable = true)
 |-- ASJC: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- subjareas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Af: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- affiliation_address_part: string (nullable = true)
 |    |    |-- affiliation_city: string (nullable = true)
 |    |    |-- affiliation_city_group: string (nullable = true)
 |    |    |-- affiliation_country: string (nullable = true)
 |    |    |-- affiliation_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- affiliation_organization: array (nullable = true)
 |    |    |    |-- 

PROJECT DETAILS:
Project identifier: proj_050_teamscience
Cluster data access level: fulldata


In [0]:
df_hyp1 = df_ani_smaller.select("Eid", "Year", "Au", "ASJC", "CitationCount")\
  .withColumn("Au", explode("Au"))\
  .withColumn("ASJC", explode("ASJC"))

df_hyp1 = df_hyp1\
  .withColumn('subjareas',
              when(df_hyp1.ASJC.between(1100, 1199), "Life Sciences")\
  .when(df_hyp1.ASJC.between(1200, 1299), "Social Sciences")\
  .when(df_hyp1.ASJC.between(1300, 1399), "Life Sciences")\
  .when(df_hyp1.ASJC.between(1400, 1499), "Social Sciences")\
  .when(df_hyp1.ASJC.between(1500, 1799), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(1800, 1899), "Social Sciences")\
  .when(df_hyp1.ASJC.between(1900, 1999), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(2000, 2099), "Social Sciences")\
  .when(df_hyp1.ASJC.between(2100, 2399), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(2400, 2499), "Life Sciences")\
  .when(df_hyp1.ASJC.between(2500, 2699), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(2700, 2799), "Health Sciences")\
  .when(df_hyp1.ASJC.between(2800, 2899), "Life Sciences")\
  .when(df_hyp1.ASJC.between(2900, 2999), "Health Sciences")\
  .when(df_hyp1.ASJC.between(3000, 3099), "Life Sciences")\
  .when(df_hyp1.ASJC.between(3100, 3199), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(3200, 3399), "Social Sciences")\
  .when(df_hyp1.ASJC.between(3400, 3699), "Health Sciences")\
  .otherwise("Multidisciplinary")
  ).groupBy("Eid", "Year", "subjareas", "CitationCount")\
  .agg(count("Au").alias("team_size"))

In [0]:
summary_df = df_hyp1.select("subjareas", "team_size")\
  .groupBy("subjareas")\
  .agg(mean("team_size").alias("mean"), stddev("team_size").alias("sd"), count("team_size").alias("sample"), percentile_approx("team_size", 0.5, 10000).alias("median"), percentile_approx("team_size", 0.25, 10000).alias("iqr_25"), percentile_approx("team_size", 0.75, 10000).alias("iqr_75"))\
  .toPandas()
summary_df["model"] = "hyp1a"
descriptives_output_df = pd.concat([descriptives_output_df, summary_df], ignore_index = True, axis = 0)

# descriptives_output_df

### Overall Summary

In order to determine if team sizes change overall across time, we will use a linear model of team size predicted by year of publication in which each article is only represented once, regardless of discipline. We will examine if the confidence interval of the slope parameter is different from zero to determine if the predictor was significant.

In [0]:
# 4.19s on default #54.96 seconds on fulldata 
# get the total number of authors by year 
df_hyp1_overall = df_ani_smaller.select("Eid", "Year", "Au", "CitationCount", explode("Au").alias("author"))\
  .groupBy("Eid", "Year", "CitationCount")\
  .agg(count("author").alias("team_size"))

# display(df_hyp1_overall)

# team size ~ year of publication on the entire dataset 
# the "features" or X must be in this feature format
vectorAssembler = VectorAssembler(inputCols = ['Year'], outputCol = "pub_year")
df_hyp1_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp1_overall)

# put in the features X and prediction Y
lr = LinearRegression(featuresCol = 'pub_year', labelCol='team_size')
lr_model = lr.fit(df_hyp1_overall)
model_output_df = modelsummary(lr_model, "hyp1a", model_output_df)

# model_output_df

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.076015   0.000330  230.638   0.000000
## -148.212392   0.660750 -224.309   0.000000
## ---
## Mean squared error:  332.658225 , RMSE:  18.238921
## Multiple R-squared: 0.003657 ,             Total iterations: 0


### Summary By Subject Area

Last, we will calculate each discipline's slope and confidence interval using a linear model of team size predicted by year of publication. Again, we will provide these slopes in a visualization for readers to examine with no official post hoc comparison. While a multilevel model would allow us to embed all disciplines within one model, no *pyspark* multilevel model implementation exists to model data at this scale.

In [0]:


for AREA in descriptives_output_df.subjareas.unique():
  if AREA == 'undefined': continue
  print(AREA)
  print("\n-------\n")
  temp_df = df_hyp1.select("subjareas", "team_size", "Year")\
    .filter(df_hyp1.subjareas == AREA)
  
  vectorAssembler = VectorAssembler(inputCols = ['Year'], outputCol = "pub_year")
  temp_df = vectorAssembler.setHandleInvalid("skip").transform(temp_df)
  lr = LinearRegression(featuresCol = 'pub_year', labelCol='team_size')
  lr_model = lr.fit(temp_df)
  model_output_df = modelsummary(lr_model, ("hyp1a_"+AREA), model_output_df)
  
# model_output_df

Health Sciences

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.161406   0.000362  445.506   0.000000
## -316.562858   0.725480 -436.350   0.000000
## ---
## Mean squared error:  148.218153 , RMSE:  12.174488
## Multiple R-squared: 0.039821 ,             Total iterations: 0
Physical Sciences

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.215910   0.001015  212.687   0.000000
## -423.821114   2.036702 -208.092   0.000000
## ---
## Mean squared error:  1308.429130 , RMSE:  36.172215
## Multiple R-squared: 0.006661 ,             Total iterations: 0
Multidisciplinary

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
#

## Research Question 1b

### Descriptive Statistics

For this research question, we will examine the impact of team size on article publication metrics for impact. Our hypothesis is that team size impacts article metrics (non-directional). We will run these analyses once for each dependent variable of citation count and weighted citation five year count. We will calculate the mean, median, and interquartile range of each publication metric by discipline. We will visualize these values instead of comparing them directly.

In [0]:
# grab Citations_5y from df_fwci
df_fwci = spark.read.format("parquet").load(fwci_parquet_path)
df_fwci.createOrReplaceTempView("df_fwci")

# display(df_fwci)

# merge this information with tables from above 
df_hyp1_overall = df_hyp1_overall.join(df_fwci.select("Eid", "Citations_5y"), "Eid", "left")

df_hyp1 = df_hyp1.join(df_fwci.select("Eid", "Citations_5y"), "Eid", "left")

In [0]:
# summarize citation count
summary_df = df_hyp1.select("subjareas", "CitationCount")\
  .groupBy("subjareas")\
  .agg(mean("CitationCount").alias("mean"), 
       stddev("CitationCount").alias("sd"), 
       count("CitationCount").alias("sample"), 
       percentile_approx("CitationCount", 0.5, 10000).alias("median"),
       percentile_approx("CitationCount", 0.25, 10000).alias("iqr_25"), 
       percentile_approx("CitationCount", 0.75, 10000).alias("iqr_75"))\
  .toPandas()
summary_df["model"] = "hyp1b_CitationCount"
descriptives_output_df = pd.concat([descriptives_output_df, summary_df], ignore_index = True, axis = 0)

# summarize citation 5y, exclude last 5 years in graph
summary_df = df_hyp1.select("subjareas", "Citations_5y")\
  .groupBy("subjareas")\
  .agg(mean("Citations_5y").alias("mean"), 
       stddev("Citations_5y").alias("sd"), 
       count("Citations_5y").alias("sample"), 
       percentile_approx("Citations_5y", 0.5, 10000).alias("median"),
       percentile_approx("Citations_5y", 0.25, 10000).alias("iqr_25"), 
       percentile_approx("Citations_5y", 0.75, 10000).alias("iqr_75"))\
  .toPandas()
summary_df["model"] = "hyp1b_Citation5y"
descriptives_output_df = pd.concat([descriptives_output_df, summary_df], ignore_index = True, axis = 0)

### Overall Summary 
To determine the impact of team size on publication metrics, we will use a linear model of publication metrics predicted by time (covariate) and team size. We will include time as a covariate, as we expect that older articles will potentially have more or different metrics because they have been around longer to cite. We will examine if the confidence interval of the team size slope parameter is different from zero.

In [0]:
# citation count ~ year of publication + team_size
# the "features" or X must be in this feature format
df_hyp1_overall = df_hyp1_overall.drop("pub_year")
vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size'], outputCol = "pub_year")
df_hyp1_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp1_overall)

# put in the features X and prediction Y
lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'CitationCount')
lr_model = lr.fit(df_hyp1_overall)
model_output_df = modelsummary(lr_model, "hyp1b_CitationCount", model_output_df)

# model_output_df

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##  -0.211645   0.002141  -98.858   0.000000
##   0.124026   0.001725   71.897   0.000000
## 445.792143   4.291649  103.874   0.000000
## ---
## Mean squared error:  13984.975948 , RMSE:  118.258090
## Multiple R-squared: 0.000976 ,             Total iterations: 0


In [0]:
# citation_5y ~ year of publication + team_size
# the "features" or X must be in this feature format
df_hyp1_overall_sub = df_hyp1_overall.drop("pub_year")\
  .filter(df_hyp1_overall.Year < 2017).where(col("Citations_5y").isNotNull())
vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size'], outputCol = "pub_year")
df_hyp1_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp1_overall_sub)

# put in the features X and prediction Y
lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'Citations_5y')
lr_model = lr.fit(df_hyp1_overall)
model_output_df = modelsummary(lr_model, "hyp1b_Citation5y", model_output_df)

# model_output_df

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.241805   0.000573  421.660   0.000000
##   0.107054   0.000515  207.680   0.000000
## -476.273394   1.146399 -415.452   0.000000
## ---
## Mean squared error:  596.380536 , RMSE:  24.420904
## Multiple R-squared: 0.020929 ,             Total iterations: 0


### Summary By Subject Area
Last, we will calculate the slope for team size for each discipline separately. We will visualize these differences and the confidence intervals.

In [0]:
for AREA in descriptives_output_df.subjareas.unique():
  if AREA == 'undefined': continue
  print(AREA)
  print("\n-------\n")
  temp_df = df_hyp1.select("subjareas", "team_size", "Year", "CitationCount", "Citations_5y")\
    .filter(df_hyp1.subjareas == AREA)
  
  vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size'], outputCol = "pub_year")
  temp_df = vectorAssembler.setHandleInvalid("skip").transform(temp_df)
  lr = LinearRegression(featuresCol = 'pub_year', labelCol='CitationCount')
  lr_model = lr.fit(temp_df)
  model_output_df = modelsummary(lr_model, ("hyp1b_CitationCount_"+AREA), model_output_df)
  
  lr = LinearRegression(featuresCol = 'pub_year', labelCol='Citations_5y')
  lr_model = lr.fit(temp_df.filter(temp_df.Year < 2017).where(col("Citations_5y").isNotNull()))
  model_output_df = modelsummary(lr_model, ("hyp1b_Citation5y_"+AREA), model_output_df)
  
# model_output_df

Health Sciences

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##  -0.085299   0.002356  -36.199   0.000000
##   0.350730   0.002759  127.111   0.000000
## 188.303109   4.715052   39.937   0.000000
## ---
## Mean squared error:  6040.835080 , RMSE:  77.722809
## Multiple R-squared: 0.003928 ,             Total iterations: 0
Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.194026   0.000787  246.644   0.000000
##   0.432626   0.001435  301.460   0.000000
## -383.261709   1.568950 -244.279   0.000000
## ---
## Mean squared error:  392.179097 , RMSE:  19.803512
## Multiple R-squared: 0.052968 ,             Total iterations: 0
Physical Sciences

-------

Note: the last rows are the information for Intercept
## -------------------------------------------

# Output Tables

Output tables for writing manuscript in R markdown.

In [0]:
# write out results section until I can get download link
pd.set_option('display.max_rows', None)
model_output_df

,model,coef,se,t,p,r2
0,hyp1a,0.076015,0.00033,230.637771,0.0,0.003657
1,hyp1a,-148.212392,0.66075,-224.309254,0.0,0.003657
2,hyp1a_Health Sciences,0.161406,0.000362,445.506137,0.0,0.039821
3,hyp1a_Health Sciences,-316.562858,0.72548,-436.349618,0.0,0.039821
4,hyp1a_Physical Sciences,0.21591,0.001015,212.686586,0.0,0.006661
5,hyp1a_Physical Sciences,-423.821114,2.036702,-208.091816,0.0,0.006661
6,hyp1a_Multidisciplinary,0.126709,0.001883,67.303367,0.0,0.016911
7,hyp1a_Multidisciplinary,-248.70446,3.784986,-65.708158,0.0,0.016911
8,hyp1a_Social Sciences,0.063064,0.000305,206.669243,0.0,0.021208
9,hyp1a_Social Sciences,-122.670931,0.612814,-200.176605,0.0,0.021208


In [0]:
descriptives_output_df

,model,subjareas,mean,sd,sample,median,iqr_25,iqr_75
0,hyp1a,Health Sciences,6.642635,13.71463,4791056,5,2,8
1,hyp1a,Physical Sciences,9.338888,35.822984,6747322,6,3,12
2,hyp1a,Multidisciplinary,5.998867,13.57461,251606,5,3,7
3,hyp1a,Social Sciences,3.976119,4.064987,1978327,3,2,5
4,hyp1a,Life Sciences,8.486903,10.266732,3824611,6,3,10
5,hyp1b_CitationCount,Health Sciences,19.830873,81.178642,4791676,5,0,19
6,hyp1b_CitationCount,Physical Sciences,20.918115,102.823374,6747232,6,1,20
7,hyp1b_CitationCount,Multidisciplinary,56.325842,250.901025,250661,12,2,39
8,hyp1b_CitationCount,Social Sciences,18.268724,89.590932,1978023,4,1,15
9,hyp1b_CitationCount,Life Sciences,28.945669,116.625923,3823854,12,3,31


EXECUTION DETAILS:
Spark version: 12.2.x-aarch64-scala2.12, 
Cluster Node Type: r6gd.2xlarge, 
Driver Node Type: r6gd.2xlarge, 
Worker Node Type: r6gd.2xlarge


DATA SNAPSHOT DETAILS:
This notebook uses the latest snapshot date for this version: "v008.20230509104116/", (09 May 2023)

 The following are the Scopus snapshot dates available for datasets at v008:
['v008.20220314092239/', 'v008.20220906063204/', 'v008.20221021151538/', 'v008.20230509104116/']
(If you want to use a different snapshot, in the first cell of your notebook, set the variable `custom_snapshot_date` to one of the options listed above (string data type))



DATA SAMPLE DETAILS:
Setting paths to run on full datasets (100% Scopus publication volume)


In [0]:
model_output_df.to_csv('/dbfs/Projects/proj_050_teamsci/model_output_200723_part1.csv')
descriptives_output_df.to_csv('/dbfs/Projects/proj_050_teamsci/descriptives_output_df_200723_part1.csv')

In [0]:
%fs ls s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci

path,name,size,modificationTime
s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci/descriptives_output_df_200723_part1.csv,descriptives_output_df_200723_part1.csv,1311,1689933426000
s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci/descriptives_output_df_200723_part2.csv,descriptives_output_df_200723_part2.csv,53,1689933341000
s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci/model_output_200723_part1.csv,model_output_200723_part1.csv,5525,1689933409000
s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci/model_output_200723_part2.csv,model_output_200723_part2.csv,8325,1689933305000
